# Scoring the extraction

*You extracted fields from a thousand documents. How many are wrong, which ones, and how would you know without a lawyer?*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omnifroodle/couchbase_notebooks/blob/main/notebooks/enrich/02_scoring_the_extraction.ipynb)
[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/omnifroodle/couchbase_notebooks?quickstart=1)

**Claim.** "Extraction accuracy" is not one number, the ground truth needs as much cleaning as the output, and the check that flags bad extractions in production is not the obvious one.
**Result.** Four fields, four definitions of correct, 80.3% to 95.0% accuracy — and 43% of the date annotations do not parse. Two cheap self-checks are tested against the answers; neither catches a quarter of the errors.
**Requires.** couchbase · llm · dataset-download
**Read** ~12 min · **Run** ~8 min · **Cost** a few cents

[`data-model/01`](../data-model/01_documents_that_learn.ipynb) pulled contract terms out of
twenty documents and wrote them to the database, and at no point did anything check whether they
were right. That is the normal state of an enrichment pipeline and it is not a comfortable one:
the fields look plausible, they query nicely, and nobody knows what fraction of them are
invented.

CUAD makes it checkable. Lawyers annotated these same contracts for these same facts, marking
the exact characters. So this notebook does the thing the last one skipped.

It also scales up. Twenty contracts is enough to *show* an extraction; it is nowhere near
enough to *measure* one, and section 5 is where that starts to matter.

**What this notebook does**

1. Extracts the same terms from 122 contracts.
2. Decides, per field, what "correct" means — three fields, three answers.
3. Scores them, and finds that the headline number is hiding something.
4. Looks at the failures.
5. Tests two ways to flag a bad extraction **without** ground truth, because production has no
   lawyer. One does nothing. The other works.
6. Writes the verdict back onto the documents, so downstream queries can avoid the shaky ones.

In [ ]:
# --- Setup. Works in a local checkout and on Colab. -------------------------
import os
import pathlib
import subprocess
import sys

# Cloned on Colab, where there is no local checkout. Override to test a fork.
REPO_URL = os.environ.get("CBNB_REPO_URL", "https://github.com/omnifroodle/couchbase_notebooks")

try:
    import cbnb
except ModuleNotFoundError:
    here = pathlib.Path.cwd()
    root = next((p for p in [here, *here.parents] if (p / "cbnb" / "__init__.py").exists()), None)
    if root is None:
        # Colab: clone the repo so the committed datasets come with it.
        subprocess.check_call(["git", "clone", "--depth", "1", "--quiet", REPO_URL, "cbnb-repo"])
        root = pathlib.Path("cbnb-repo").resolve()
    sys.path.insert(0, str(root))
    import cbnb

settings = cbnb.bootstrap(requires=["couchbase", "llm", "dataset-download"])

## 1. Enough contracts to measure something

The committed sample is twenty contracts. At that size a field that is 85% right shows three
failures, and three is not a number you can reason about — remove one and the accuracy moves
five points.

So this notebook downloads CUAD and takes every contract long enough to need retrieval and
annotated richly enough to ask questions of. That is a 40 MB download the first time, and it is
the cheapest part of measuring anything.

In [2]:
from cbnb.datasets import load_cuad

cuad = load_cuad(full=True)
contracts = cuad.contracts
print(cuad.summary())

122 contracts, 41 types, 41 clause categories, 3,015 annotated spans; 64% of questions have no answer in their contract


## 2. Extract

The same fields and the same prompt as
[`data-model/01`](../data-model/01_documents_that_learn.ipynb), so this measures that notebook's
work rather than a new attempt at it.

In [3]:
from pydantic import BaseModel, Field

from cbnb.llm import LLM


class Terms(BaseModel):
    title: str = Field(description="the contract's own title, e.g. 'DISTRIBUTOR AGREEMENT'")
    parties: list[str] = Field(description="the legal entities that signed it")
    governing_law: str = Field(description="the jurisdiction whose law governs, e.g. 'Texas'; empty if not stated")
    agreement_date: str = Field(description="the date signed, as YYYY-MM-DD; empty if not stated")


SYSTEM = ("You read commercial contracts and extract their key terms. Use only what the text "
          "says. Leave a field empty or null rather than guessing.")

llm = LLM()


def extract(row, system=SYSTEM):
    return llm.structured(f"Contract:\n\n{row.text}", Terms, system=system, max_tokens=500)


rows = list(contracts.itertuples(index=False))
extracted = llm.map(rows, extract, max_workers=8)
print(f"\nextracted terms from {len(extracted)} contracts")

  25/122

  50/122

  75/122

  100/122

  122/122



extracted terms from 122 contracts


## 3. What does "correct" mean?

Here is the part that generic "extraction accuracy" hides. The model returns a *value*; the
lawyer marked a *span of the contract*. Comparing them is a different problem for every field.

| field | the model says | the lawyer marked | so the test is |
| --- | --- | --- | --- |
| `governing_law` | `"Texas"` | *"This Agreement shall be governed by the laws of the State of Texas, without reference to…"* | is the value **inside** the marked sentence? |
| `agreement_date` | `"2018-07-20"` | *"July 20, 2018"* | do both **parse to the same date**? |
| `title` | `"CONSULTING AGREEMENT"` | *"CONSULTING AGREEMENT"* | do they **match** once punctuation and case are normalised? |
| `parties` | `["Kiromic, Inc.", "Gianluca Rotino"]` | several spans, one per party | does **every** extracted party appear among them? |

Four fields, four rules. Choose them differently and the numbers move — which is why they
belong here, in the open, rather than inside a scoring helper.

In [4]:
import re

import pandas as pd


def squash(value):
    """Lowercase, strip punctuation, collapse whitespace. Both sides, always."""
    return " ".join(re.sub(r"[^a-z0-9 ]", " ", str(value).lower()).split())


def marked_spans(contract_id, category):
    hit = cuad.spans[(cuad.spans.contract_id == contract_id) & (cuad.spans.category == category)]
    return hit.text.tolist()


def inside_span(value, spans):
    """For a value the lawyer marked a whole sentence around."""
    v = squash(value)
    return bool(v) and any(v in squash(s) for s in spans)


def same_date(value, spans):
    """Both sides parsed as dates. Returns None when the *annotation* will not parse."""
    truth = pd.to_datetime(spans[0], errors="coerce")
    if pd.isna(truth):
        return None
    mine = pd.to_datetime(value, errors="coerce")
    return bool(pd.notna(mine) and mine.date() == truth.date())


def matches_span(value, spans):
    return squash(value) in [squash(s) for s in spans]


def all_present(values, spans):
    blob = " ".join(squash(s) for s in spans)
    return bool(values) and all(squash(v) in blob for v in values)


RULES = {
    "governing_law": ("Governing Law", inside_span),
    "agreement_date": ("Agreement Date", same_date),
    "title": ("Document Name", matches_span),
    "parties": ("Parties", all_present),
}

In [5]:
scores = []
for row, terms in zip(rows, extracted):
    for field, (category, rule) in RULES.items():
        spans = marked_spans(row.contract_id, category)
        if not spans:
            continue                      # the lawyers found no such clause: nothing to score
        scores.append({
            "contract_id": row.contract_id,
            "field": field,
            "value": getattr(terms, field),
            "correct": rule(getattr(terms, field), spans),
        })

scored = pd.DataFrame(scores)
summary = scored.groupby("field").correct.agg(
    annotated="size",
    scored=lambda s: s.notna().sum(),
    correct=lambda s: s.sum(),
)
summary["accuracy"] = (summary.correct / summary.scored * 100).round(1)
summary

,annotated,scored,correct,accuracy
field,,,,
agreement_date,118,67,59,88.1
governing_law,119,119,113,95.0
parties,122,122,98,80.3
title,122,122,107,87.7


## 4. The number that is hiding something

Look at `scored` against `annotated` for `agreement_date`.

The date rule returns `None` when **the annotation itself** will not parse — and it does that
often. Those rows are not failures of the model; they were dropped before the model was judged,
and the accuracy printed above is over the survivors.

A field that reports 100% on half its rows is not a field that is 100% right.

In [6]:
unparseable = []
for row, terms in zip(rows, extracted):
    spans = marked_spans(row.contract_id, "Agreement Date")
    if spans and pd.isna(pd.to_datetime(spans[0], errors="coerce")):
        unparseable.append({"contract_id": row.contract_id, "annotation": spans[0][:60],
                            "model_said": terms.agreement_date})

print(f"{len(unparseable)} of {summary.loc['agreement_date', 'annotated']} date annotations "
      f"could not be parsed\n")
pd.DataFrame(unparseable).head(10)

51 of 118 date annotations could not be parsed



,contract_id,annotation,model_said
0,2,17t h day of February 2016,2016-02-17
1,5,31 day of July 2019,2019-07-31
2,13,"December 3rd, 2018 (",2018-12-03
3,14,"1st day of January, 1999",1999-01-01
4,15,"____ day of May, 2000",2000-05-01
5,16,"[ ] day of [ ], 2020 (",
6,17,20t h day of December 2018,2018-12-20
7,18,"25th day of May, 1999.",1999-05-25
8,20,"3rd day of November, 2005",2005-11-03
9,22,"23rd day of December, 2003",2003-12-23


Some are formats a date parser reasonably declines — *"23rd day of December, 2003"*, or
*"20t h day of December 2018"* with an OCR space in the middle of the number. Others are
contract templates whose date was never filled in: *"____ day of May, 2000"*, and
*"[ ] day of [ ], 2020 ("*. Those are not extraction failures at all — there is no date in the
document to find, and on one of them the model correctly returned nothing.

**The ground truth needs as much cleaning as the output.** That is the unglamorous half of
evaluation, and skipping it is how a pipeline reports a number nobody should believe. Every one
of those rows is a decision — parse harder, score by hand, or exclude and say so.

## 5. Where it actually fails

The fields do not fail equally, and that turns out to matter more than the average.

In [7]:
wrong = scored[scored.correct == False]          # noqa: E712 - None is a third state here
print(wrong.field.value_counts().to_string(), "\n")

for row in wrong[wrong.field == "title"].head(4).itertuples(index=False):
    marked = marked_spans(row.contract_id, "Document Name")
    print(f"contract {row.contract_id}")
    print(f"  model  : {row.value!r}")
    print(f"  lawyer : {marked[0]!r}\n")

field
parties           24
title             15
agreement_date     8
governing_law      6 

contract 2
  model  : 'STRATEGIC ALLIANCE AGREEMENT "EDGE-FTE"'
  lawyer : 'STRATEGIC ALLIANCE AGREEMENT'

contract 10
  model  : 'PSITECH CORPORATION WEBSITE CONTENT LICENSE AGREEMENT'
  lawyer : 'WEBSITE CONTENT LICENSE AGREEMENT'

contract 16
  model  : 'TRADEMARK LICENSE AGREEMENT'
  lawyer : 'FORM OF TRADEMARK LICENSE AGREEMENT'

contract 23
  model  : '2008 Sponsorship Agreement- Renewal Sponsor'
  lawyer : 'SPONSORSHIP AGREEMENT'



## 6. Production has no lawyer

Everything above needed annotations. On your own documents there are none, and the question
becomes: **can the pipeline tell you which of its own extractions to distrust?**

Two candidate flags, both computable at extraction time with no ground truth at all.

**Is the value verbatim in the document?** The obvious one. An extracted value that does not
appear in the text it came from is invented.

In [8]:
text_by_id = contracts.set_index("contract_id").text

verbatim = []
for row in scored[scored.field.isin(["governing_law", "title"])].itertuples(index=False):
    document = squash(text_by_id[row.contract_id])
    verbatim.append({"field": row.field, "correct": row.correct,
                     "in_document": squash(row.value) in document})

verbatim = pd.DataFrame(verbatim).dropna(subset=["correct"])
pd.crosstab(verbatim.in_document, verbatim.correct, margins=True)

correct,False,True,All
in_document,,,
False,4,0,4
True,17,220,237
All,21,220,241


Read that table carefully, because it is more interesting than "it works" or "it doesn't".

The check fires **four times out of 241, and every one of those four is wrong**. Perfect
precision — when a value is not in the document it came from, something has gone wrong, every
time. And it still misses **17 of the 21 errors**, because the other seventeen are values that
*are* in the document, just not the right ones.

The reason is the prompt: we told the model to use only what the text says, and it complied.
Verbatim grounding catches invention from nothing. It cannot catch copying the wrong sentence,
which is what actually goes wrong here.

**Do two different prompts agree?** The second candidate. Ask for the same fields in a different
voice, and treat disagreement as doubt. It costs a second extraction, which is the honest price
of not having ground truth.

In [9]:
PARALEGAL = ("You are a paralegal indexing a contract database. Record the governing "
             "jurisdiction and the document's title exactly as the contract states them. "
             "Leave a field empty rather than guessing.")

second = llm.map(rows, lambda r: extract(r, system=PARALEGAL), max_workers=8)

agreement = []
for row, first, other in zip(rows, extracted, second):
    for field in ("governing_law", "title"):
        category, rule = RULES[field]
        spans = marked_spans(row.contract_id, category)
        if not spans:
            continue
        agreement.append({
            "field": field,
            "agree": squash(getattr(first, field)) == squash(getattr(other, field)),
            "correct": rule(getattr(first, field), spans),
        })

agreement = pd.DataFrame(agreement)
pd.crosstab([agreement.field, agreement.agree], agreement.correct, margins=True)

  25/122

  50/122

  75/122

  100/122

  122/122

correct              False  True  All
field         agree                  
governing_law False      3    11   14
              True       3   102  105
title         False      2     2    4
              True      13   105  118
All                     21   220  241

In [10]:
for field in ("governing_law", "title"):
    sub = agreement[agreement.field == field]
    agree, differ = sub[sub.agree], sub[~sub.agree]
    line = f"{field:15} overall {sub.correct.mean():.0%}"
    if len(agree):
        line += f" | when the prompts agree {agree.correct.mean():.0%} (n={len(agree)})"
    if len(differ):
        line += f" | when they differ {differ.correct.mean():.0%} (n={len(differ)})"
    print(line)

governing_law   overall 95% | when the prompts agree 97% (n=105) | when they differ 79% (n=14)
title           overall 88% | when the prompts agree 89% (n=118) | when they differ 50% (n=4)


Disagreement really is enriched for errors. Half the disputed titles are wrong, against a base
rate of roughly one in nine; disputed jurisdictions are wrong about a fifth of the time against
one in twenty. Pointing a reviewer at the disagreements is better than pointing them at random
rows.

But look at how rarely it fires. Four disputed titles. Fourteen disputed jurisdictions. Between
them they contain **5 of the 21 errors** — so a reviewer working that queue still never sees
three quarters of what is wrong.

The cause is worth naming: **two prompts to the same model mostly agree, because they share its
blind spots.** Rewording the instruction does not produce an independent second opinion. A
second *model* would, and that is the version worth paying for — this table is the evidence for
spending the money rather than an argument that the cheap version suffices.

So the practical position is uncomfortable and correct. Both checks are worth running: they are
cheap, one of them never cries wolf, and together they put a reviewer in front of the rows most
likely to be wrong. **Neither is assurance.** The only thing in this notebook that actually told
us `parties` is the weakest field at 80% was the annotated sample.

## 7. Write the verdict onto the document

A quality flag is only useful if a query can see it. It belongs on the derived document beside
the value it describes, in the same shape
[`data-model/01`](../data-model/01_documents_that_learn.ipynb) established.

In [11]:
import couchbase.subdocument as SD

from cbnb.couchbase_io import connect, ensure_collection, upsert_docs

BUCKET, SCOPE, COLLECTION = settings.cb_bucket, "contract_intelligence", "extraction_quality"

cluster = connect(settings)
quality = ensure_collection(cluster, BUCKET, SCOPE, COLLECTION)

records = {}
for row, first, other in zip(rows, extracted, second):
    confident = squash(first.title) == squash(other.title)
    records[f"contract::{row.contract_id}::quality"] = {
        "type": COLLECTION,
        "of": f"contract::{row.contract_id}",
        "contract_id": int(row.contract_id),
        "title": first.title,
        "title_confidence": "high" if confident else "review",
        "checked_by": "two-prompt agreement",
        "models": [llm.model, llm.model],
    }

upsert_docs(quality, records, progress=False)
flagged = sum(1 for r in records.values() if r["title_confidence"] == "review")
print(f"{len(records)} quality records written; {flagged} titles flagged for review "
      f"({flagged / len(records):.0%})")

122 quality records written; 4 titles flagged for review (3%)


In [12]:
list(cluster.query(f"""
    SELECT title_confidence, COUNT(*) AS contracts
    FROM `{BUCKET}`.`{SCOPE}`.`{COLLECTION}`
    GROUP BY title_confidence
"""))

[{'contracts': 118, 'title_confidence': 'high'},
 {'contracts': 4, 'title_confidence': 'review'}]

## What to take from this

**"Extraction accuracy" is four numbers and four arguments.** Each field needs its own definition
of correct, and every one of those definitions is a choice that moves the result. A single
percentage in a status report has had all of that decided for you by whoever wrote the scorer.

**Clean the ground truth too.** A sizeable share of the date annotations here would not parse,
including one that is an unfilled blank in a contract template. Those rows silently left the
denominator. Any evaluation that reports `n` without reporting *how many it could not score* is
telling you less than it appears to.

**A self-check is triage, not assurance.** The two tested here are cheap and both are worth
running — verbatim grounding is never wrong when it fires, and prompt disagreement concentrates
errors well above the base rate. Between them they still leave most of the failures undetected.
Anyone reporting extraction quality from a self-check alone is reporting a number they have not
earned.

**Two prompts to one model is not a second opinion.** They share its blind spots and therefore
mostly agree. If disagreement is going to be your flag, pay for a second model — the weakness of
the signal here is the argument for it.

## Where to take this

- **Route the flagged ones.** A `review` verdict is only worth writing if something consumes it —
  a queue, a second pass with a stronger model, or a query that excludes low-confidence rows from
  a report.
- **Try a second model rather than a second prompt.** Two prompts to the same model share its
  blind spots; two different models agreeing is a stronger signal, and `cbnb.llm` will point at
  either.
- **Score the fields this notebook skipped.** `term_years` and `auto_renews` have no clean CUAD
  equivalent, which is its own finding: the facts most useful for filtering are often the ones
  nobody annotated.
- **Re-run it when the model changes.** The accuracy here belongs to one model on one day. The
  scoring harness is the durable part, which is the argument for building it at all.

> **On Couchbase AI Data Plane** — the extraction loop is the part a managed service can take
> over. The scoring is not: deciding what "correct" means for your fields, and finding out which
> of them need a quality gate, stays your problem no matter who runs the model. Store the verdict
> next to the value either way.

In [13]:
# The quality records this notebook wrote. Uncomment to remove just those.
# list(cluster.query(f'DELETE FROM `{BUCKET}`.`{SCOPE}`.`{COLLECTION}`'))